# 04SMOTE_5-fold_model_train 노트북 목표
1. MLP 구성 후보군을 설정한다.
2. validation 성능을 기준으로 사용할 임계값(threshold)을 정한다.
3. MLP 구성을 찾는 도중 데이터 누수를 방지하기 위해, PCA, 메타데이터 스케일링, SMOTE, MLP 학습을 5-Fold 교차검증의 각 fold 내부에서 수행한다.
4. 각 MLP 설정마다 5-fold 교차검증을 수행하고, 각 fold 안에서 scaler, PCA, SMOTE를 적용한 뒤 모델을 학습한다. 이후 fold별 검증 성능을 평균내고, 평균 F1-score가 가장 높은 설정을 최적 설정으로 선택한다.
5. 이후 해당 모델을 저장한다.

## 1. 학습용 라이브러리 로드

In [21]:
import json
import os
from itertools import product

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

## 2. 데이터 로드 및 입력/라벨 분리

- 02번에서 생성한 `reviews_embeddings_extract.csv`를 사용한다.
- PCA와 메타데이터 스케일링을 CV fold 내부에서 학습하기 위해 원본 임베딩 768차원 상태로 로드한다.
- train / validation / test를 70% / 15% / 15%로 분리한다.
- `rating`은 이벤트 판별 feature에서 제외한다.

In [22]:
SEED = 42
TARGET_COL = 'label'
TEXT_COL = 'cleaned_review_text'

raw_df = pd.read_csv('csv/reviews_embeddings_extract.csv')

emb_cols = [f'kcbert_{i}' for i in range(768)]
meta_cols = ['text_length', 'emoji_count', 'photo_count']
feature_cols = emb_cols + meta_cols

X_all = raw_df[feature_cols].copy()
y_all = raw_df[TARGET_COL].astype(int)

X_train_all, X_temp, y_train_all, y_temp = train_test_split(
    X_all,
    y_all,
    test_size=0.3,
    random_state=SEED,
    stratify=y_all,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=SEED,
    stratify=y_temp,
)

print('전체 데이터:', raw_df.shape)
print('학습용 데이터:', X_train_all.shape)
print('검증용 데이터:', X_val.shape)
print('테스트용 데이터:', X_test.shape)
print('입력 feature 수:', len(feature_cols))
print('사용 메타데이터 컬럼:', meta_cols)
print('학습 데이터 라벨 분포, 0=일반 리뷰, 1=이벤트 리뷰')
print(y_train_all.value_counts().sort_index().to_dict())

전체 데이터: (8841, 782)
학습용 데이터: (6188, 771)
검증용 데이터: (1326, 771)
테스트용 데이터: (1327, 771)
입력 feature 수: 771
사용 메타데이터 컬럼: ['text_length', 'emoji_count', 'photo_count']
학습 데이터 라벨 분포, 0=일반 리뷰, 1=이벤트 리뷰
{0: 3983, 1: 2205}


/var/folders/wq/j2lqqp7n33dgrwj1r4q0nl9m0000gn/T/ipykernel_99311/2170499664.py:5: DtypeWarning: Columns (0: store_name) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv('csv/reviews_embeddings_extract.csv')


## 3. 후보 MLP 모델 정의

- 최종 분류 모델은 Scikit-learn `MLPClassifier`를 사용한다.
- 입력 feature는 KcBERT 임베딩 768차원과 `text_length`, `emoji_count`, `photo_count` 메타데이터이다.
- `ColumnTransformer`로 KcBERT 임베딩에는 PCA를, 메타데이터에는 StandardScaler를 적용한 뒤 하나의 feature로 결합한다.
- PCA, StandardScaler, SMOTE는 데이터 누수를 막기 위해 각 CV fold의 학습 데이터 안에서만 fit한다.
- 하나의 MLP 설정만 고정하지 않고 여러 하이퍼파라미터 조합을 비교한다.

### 주요 개념 정리
- 5-Fold CV: train 데이터를 5개로 나누어, 매번 4개로 학습하고 1개로 검증하는 방식이다.
- PCA: KcBERT 768차원 임베딩을 정보량을 최대한 유지하면서 더 적은 차원으로 줄이는 방법이다.
- StandardScaler: 단위가 다른 메타데이터를 평균 0, 표준편차 1 기준으로 맞추는 방법이다.
- SMOTE: 소수 클래스 샘플을 보간해 클래스 불균형을 보정하는 방법이다.
- MLP: 여러 층의 뉴런으로 구성된 신경망 분류기이다.

### 비교할 MLP 설정
- hidden layer: 은닉층 구조와 뉴런 수
- optimizer/solver: 가중치를 업데이트하는 방식 (`adam`, `sgd`)
- learning rate: 한 번에 가중치를 얼마나 크게 수정할지 정하는 값
- L2 alpha: 과적합을 줄이기 위한 정규화 강도
- activation: 은닉층에서 사용하는 비선형 함수 (`relu`, `tanh`)
- batch size: 한 번에 묶어서 학습하는 데이터 개수
- PCA 보존율: KcBERT 임베딩의 분산 정보를 얼마나 보존할지 정하는 값
- SMOTE k_neighbors: 소수 클래스 샘플 생성 시 참고할 이웃 수

### 탐색 범위
- hidden layer: `(32,)`, `(64,)`, `(128,)`, `(128, 64)`
- optimizer/solver: `adam`, `sgd`
- learning rate: `1e-3`, `3e-4`
- L2 alpha: `1e-4`, `1e-3`, `1e-2`
- activation: `relu`, `tanh`
- batch size: `32`, `64`
- PCA 보존율: `0.90`
- SMOTE k_neighbors: `5`

총 `192개` 후보 설정을 5-Fold CV로 평가하므로 `960개` fold 결과를 비교한다.

### 설정 선택 기준
- 각 후보 설정마다 5-Fold CV를 수행한다.
- fold별 F1-score, PR-AUC, precision, recall, ROC-AUC를 계산한다.
- 후보별 5개 fold 성능을 평균낸다.
- 평균 F1-score가 가장 높은 설정을 우선 선택하고, 동률이면 PR-AUC를 보조 기준으로 사용한다.


In [23]:

hidden_layer_grid = [(32,), (64,), (128,), (128, 64)]
solver_grid = ['adam', 'sgd']
learning_rate_grid = [1e-3, 3e-4]
alpha_grid = [1e-4, 1e-3, 1e-2]
activation_grid = ['relu', 'tanh']
batch_size_grid = [32, 64]
pca_n_components_grid = [0.90]
smote_k_neighbors_grid = [5]
N_SPLITS = 5

model_configs = []
for hidden_layer_sizes, solver, learning_rate_init, alpha, activation, batch_size, pca_n_components, smote_k_neighbors in product(
    hidden_layer_grid,
    solver_grid,
    learning_rate_grid,
    alpha_grid,
    activation_grid,
    batch_size_grid,
    pca_n_components_grid,
    smote_k_neighbors_grid,
):
    hidden_name = '_'.join(str(size) for size in hidden_layer_sizes)
    pca_name = str(pca_n_components).replace('.', 'p')
    model_configs.append({
        'name': f"mlp_h{hidden_name}_{activation}_{solver}_lr{learning_rate_init:g}_alpha{alpha:g}_bs{batch_size}_pca{pca_name}_smote{smote_k_neighbors}",
        'hidden_layer_sizes': hidden_layer_sizes,
        'solver': solver,
        'learning_rate_init': learning_rate_init,
        'alpha': alpha,
        'activation': activation,
        'batch_size': batch_size,
        'pca_n_components': pca_n_components,
        'smote_k_neighbors': smote_k_neighbors,
    })

CV_GRID_SIZE = len(model_configs)
MODEL_CONFIG_BY_NAME = {config['name']: config for config in model_configs}
print(f'CV 후보 설정 수: {CV_GRID_SIZE}, fold 수: {N_SPLITS}, 총 fold 평가 수: {CV_GRID_SIZE * N_SPLITS}')

PROPOSED_MLP_OUTPUTS = {
    'model': 'outputs/proposed_mlp_final_model.joblib',
    'selected_config': 'outputs/proposed_mlp_selected_config.json',
    'metrics_csv': 'outputs/proposed_mlp_metrics.csv',
    'cv_summary': 'outputs/proposed_mlp_cv_summary.csv',
}
FORCE_RERUN_CV = False

def selected_config_to_model_config(selected: dict) -> dict:
    selected_model = selected['selected_model']
    if selected_model in MODEL_CONFIG_BY_NAME:
        return MODEL_CONFIG_BY_NAME[selected_model].copy()

    return {
        'name': selected_model,
        'hidden_layer_sizes': tuple(selected['hidden_layer_sizes']),
        'solver': selected['solver'],
        'learning_rate_init': selected['learning_rate_init'],
        'alpha': selected['alpha'],
        'activation': selected['activation'],
        'batch_size': selected['batch_size'],
        'pca_n_components': selected['pca_n_components'],
        'smote_k_neighbors': selected['smote_k_neighbors'],
    }


def build_selected_config(best_config: dict, best_summary=None) -> dict:
    selected = {
        'selected_model': best_config['name'],
        'hidden_layer_sizes': list(best_config['hidden_layer_sizes']),
        'solver': best_config['solver'],
        'learning_rate_init': best_config['learning_rate_init'],
        'alpha': best_config['alpha'],
        'activation': best_config['activation'],
        'batch_size': best_config['batch_size'],
        'pca_n_components': best_config['pca_n_components'],
        'smote_k_neighbors': best_config['smote_k_neighbors'],
        'n_splits': N_SPLITS,
        'cv_grid_size': CV_GRID_SIZE,
        'selection_rule': 'highest mean_f1, then mean_pr_auc on leakage-safe 5-fold CV',
        'excluded_features': ['rating'],
    }

    if best_summary is not None:
        for source_col, target_key in {
            'mean_f1': 'cv_mean_f1',
            'std_f1': 'cv_std_f1',
            'mean_pr_auc': 'cv_mean_pr_auc',
            'std_pr_auc': 'cv_std_pr_auc',
            'mean_precision': 'cv_mean_precision',
            'mean_recall': 'cv_mean_recall',
            'mean_roc_auc': 'cv_mean_roc_auc',
        }.items():
            if source_col in best_summary and pd.notna(best_summary[source_col]):
                selected[target_key] = float(best_summary[source_col])

    return selected


def selected_config_to_summary_row(selected: dict) -> dict:
    return {
        'model': selected['selected_model'],
        'hidden_layer_sizes': str(tuple(selected['hidden_layer_sizes'])),
        'solver': selected['solver'],
        'learning_rate_init': selected['learning_rate_init'],
        'alpha': selected['alpha'],
        'activation': selected['activation'],
        'batch_size': selected['batch_size'],
        'pca_n_components': selected['pca_n_components'],
        'smote_k_neighbors': selected['smote_k_neighbors'],
        'mean_f1': selected.get('cv_mean_f1'),
        'std_f1': selected.get('cv_std_f1'),
        'mean_pr_auc': selected.get('cv_mean_pr_auc'),
        'std_pr_auc': selected.get('cv_std_pr_auc'),
        'mean_precision': selected.get('cv_mean_precision'),
        'mean_recall': selected.get('cv_mean_recall'),
        'mean_roc_auc': selected.get('cv_mean_roc_auc'),
        'cache_note': 'selected_config.json에서 불러온 최적 설정',
    }


def make_preprocessor(random_state: int, pca_n_components: float, use_emb=True, use_meta=True):
    transformers = []
    if use_emb:
        transformers.append(('kcbert_pca', PCA(n_components=pca_n_components, random_state=random_state), emb_cols))
    if use_meta:
        transformers.append(('metadata_scaler', StandardScaler(), meta_cols))

    return ColumnTransformer(transformers=transformers, remainder='drop')


def make_model(config: dict, random_state: int) -> ImbPipeline:
    return ImbPipeline([
        ('preprocess', make_preprocessor(
            random_state=random_state,
            pca_n_components=config.get('pca_n_components', 0.90),
            use_emb=True,
            use_meta=True,
        )),
        ('smote', SMOTE(
            random_state=random_state,
            k_neighbors=config.get('smote_k_neighbors', 5),
        )),
        ('mlp', MLPClassifier(
            hidden_layer_sizes=config['hidden_layer_sizes'],
            activation=config.get('activation', 'relu'),
            solver=config['solver'],
            alpha=config['alpha'],
            batch_size=config.get('batch_size', 64),
            learning_rate_init=config['learning_rate_init'],
            max_iter=200,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=10,
            random_state=random_state,
        )),
    ])


CV 후보 설정 수: 192, fold 수: 5, 총 fold 평가 수: 960



## 4. SMOTE + 5-Fold 교차검증 (**오래 걸릴 수 있음**)

- train 데이터를 5개 fold로 나누고, 각 반복에서 4개 fold는 학습용, 1개 fold는 검증용으로 사용한다.
- 앞에서 정의한 192개 MLP 후보 설정을 모두 같은 방식으로 평가한다.
- 각 fold에서는 학습 fold에만 PCA, StandardScaler, SMOTE를 적용한다.
- 검증 fold에는 학습 fold에서 만든 변환 기준만 적용하고, SMOTE는 적용하지 않는다.
- `proposed_mlp_selected_config.json`이 있으면 오래 걸리는 CV 탐색은 건너뛰고 저장된 최적 설정을 사용한다.

### 평가 지표
- `f1`: precision과 recall의 균형을 보는 지표
- `pr_auc`: 이벤트 리뷰처럼 관심 클래스 탐지 성능을 볼 때 유용한 지표
- `precision`: 이벤트 리뷰라고 예측한 것 중 실제 이벤트 리뷰의 비율
- `recall`: 실제 이벤트 리뷰 중 모델이 잡아낸 비율
- `roc_auc`: 전체적인 이진 분류 구분 능력

### Fold 내부에서만 처리하는 이유
- PCA와 StandardScaler는 평균, 분산, 주성분 구조를 학습한다.
- SMOTE는 소수 클래스 샘플을 참고해 합성 샘플을 만든다.
- 이를 전체 데이터에 먼저 적용하면 validation/test 정보가 학습 과정에 섞일 수 있다.
- 그래서 전처리와 SMOTE는 학습 fold에만 적용하고, 검증 fold와 최종 validation/test는 원본 분포로 평가한다.


In [24]:
cached_selected_config = None
if os.path.exists(PROPOSED_MLP_OUTPUTS['selected_config']) and not FORCE_RERUN_CV:
    with open(PROPOSED_MLP_OUTPUTS['selected_config'], 'r', encoding='utf-8') as f:
        cached_selected_config = json.load(f)
    can_reuse_cv_config = True
    print(f"기존 최적 MLP 설정을 재사용합니다: {PROPOSED_MLP_OUTPUTS['selected_config']}")
else:
    can_reuse_cv_config = False
    print('저장된 최적 설정이 없어 5-Fold CV를 실행합니다.')

can_reuse_cv_config


기존 최적 MLP 설정을 재사용합니다: outputs/proposed_mlp_selected_config.json


True

In [32]:
if can_reuse_cv_config:
    if os.path.exists(PROPOSED_MLP_OUTPUTS['cv_summary']):
        cv_summary = pd.read_csv(PROPOSED_MLP_OUTPUTS['cv_summary'])
        print('기존 04번 CV summary를 재사용합니다.')
    else:
        cv_summary = pd.DataFrame([selected_config_to_summary_row(cached_selected_config)])
        print('selected_config만 재사용합니다. CV summary 파일은 없으므로 선택 설정만 표시합니다.')
else:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    cv_rows = []

    for config in model_configs:
        for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_all, y_train_all), start=1):
            X_fold_train = X_train_all.iloc[train_idx]
            y_fold_train = y_train_all.iloc[train_idx]
            X_fold_valid = X_train_all.iloc[valid_idx]
            y_fold_valid = y_train_all.iloc[valid_idx]

            model = make_model(config, random_state=SEED + fold)
            model.fit(X_fold_train, y_fold_train)

            valid_prob = model.predict_proba(X_fold_valid)[:, 1]
            valid_pred = (valid_prob >= 0.5).astype(int)

            majority_count = int(y_fold_train.value_counts().max())
            cv_rows.append({
                'model': config['name'],
                'fold': fold,
                'hidden_layer_sizes': str(config['hidden_layer_sizes']),
                'solver': config['solver'],
                'learning_rate_init': config['learning_rate_init'],
                'alpha': config['alpha'],
                'activation': config['activation'],
                'batch_size': config['batch_size'],
                'pca_n_components': config['pca_n_components'],
                'smote_k_neighbors': config['smote_k_neighbors'],
                'train_before_label_0': int((y_fold_train == 0).sum()),
                'train_before_label_1': int((y_fold_train == 1).sum()),
                'train_after_smote_label_0': majority_count,
                'train_after_smote_label_1': majority_count,
                'valid_label_0': int((y_fold_valid == 0).sum()),
                'valid_label_1': int((y_fold_valid == 1).sum()),
                'f1': float(f1_score(y_fold_valid, valid_pred)),
                'pr_auc': float(average_precision_score(y_fold_valid, valid_prob)),
                'precision': float(precision_score(y_fold_valid, valid_pred, zero_division=0)),
                'recall': float(recall_score(y_fold_valid, valid_pred, zero_division=0)),
                'roc_auc': float(roc_auc_score(y_fold_valid, valid_prob)),
            })

    cv_results = pd.DataFrame(cv_rows)
    cv_summary = (
        cv_results.groupby([
            'model',
            'hidden_layer_sizes',
            'solver',
            'learning_rate_init',
            'alpha',
            'activation',
            'batch_size',
            'pca_n_components',
            'smote_k_neighbors',
        ])
        .agg(
            mean_f1=('f1', 'mean'),
            std_f1=('f1', 'std'),
            mean_pr_auc=('pr_auc', 'mean'),
            std_pr_auc=('pr_auc', 'std'),
            mean_precision=('precision', 'mean'),
            mean_recall=('recall', 'mean'),
            mean_roc_auc=('roc_auc', 'mean'),
        )
        .reset_index()
        .sort_values(['mean_f1', 'mean_pr_auc'], ascending=False)
    )

    cv_summary.to_csv(PROPOSED_MLP_OUTPUTS['cv_summary'], index=False, encoding='utf-8-sig')

cv_summary_display = cv_summary[[
    'model',
    'hidden_layer_sizes',
    'solver',
    'learning_rate_init',
    'alpha',
    'activation',
    'batch_size',
    'mean_f1',
    'mean_pr_auc',
    'mean_precision',
    'mean_recall',
]]

display(cv_summary_display.round({
    'mean_f1': 4,
    'mean_pr_auc': 4,
    'mean_precision': 4,
    'mean_recall': 4,
}))


기존 04번 CV summary를 재사용합니다.


,model,hidden_layer_sizes,solver,learning_rate_init,alpha,activation,batch_size,mean_f1,mean_pr_auc,mean_precision,mean_recall
0,mlp_h32_tanh_sgd_lr0.001_alpha0.01_bs64_pca0p9...,"(32,)",sgd,0.0010,0.0100,tanh,64,0.4638,0.4169,0.4081,0.5374
1,mlp_h32_tanh_sgd_lr0.001_alpha0.001_bs64_pca0p...,"(32,)",sgd,0.0010,0.0010,tanh,64,0.4638,0.4168,0.4081,0.5374
2,mlp_h32_tanh_sgd_lr0.001_alpha0.0001_bs64_pca0...,"(32,)",sgd,0.0010,0.0001,tanh,64,0.4625,0.4146,0.4062,0.5374
3,mlp_h64_tanh_sgd_lr0.0003_alpha0.01_bs64_pca0p...,"(64,)",sgd,0.0003,0.0100,tanh,64,0.4617,0.4140,0.3995,0.5469
4,mlp_h64_tanh_sgd_lr0.0003_alpha0.0001_bs64_pca...,"(64,)",sgd,0.0003,0.0001,tanh,64,0.4605,0.4151,0.4000,0.5429
...,...,...,...,...,...,...,...,...,...,...,...
187,mlp_h128_64_tanh_adam_lr0.001_alpha0.01_bs32_p...,"(128, 64)",adam,0.0010,0.0100,tanh,32,0.3985,0.3939,0.3956,0.4018
188,mlp_h128_64_relu_sgd_lr0.001_alpha0.0001_bs64_...,"(128, 64)",sgd,0.0010,0.0001,relu,64,0.3984,0.4065,0.3965,0.4005
189,mlp_h128_64_relu_adam_lr0.0003_alpha0.01_bs32_...,"(128, 64)",adam,0.0003,0.0100,relu,32,0.3973,0.4111,0.4053,0.3909
190,mlp_h128_64_relu_adam_lr0.0003_alpha0.001_bs32...,"(128, 64)",adam,0.0003,0.0010,relu,32,0.3952,0.4085,0.4035,0.3882



## 5. 최종 모델 학습 및 Validation/Test 평가

- CV를 실행한 경우에는 평균 F1-score와 PR-AUC 기준으로 가장 좋은 MLP 후보를 선택한다.
- 이미 `proposed_mlp_selected_config.json`이 있으면 저장된 최적 설정을 사용해 CV 탐색을 건너뛴다.
- 선택된 모델 구조를 사용해 전체 train 데이터로 최종 모델을 다시 학습한다.
- threshold는 validation 데이터에서만 조정하며, test 데이터는 최종 일반화 성능 확인용으로만 사용한다.
- 최종 학습 단계에서도 train 데이터에만 SMOTE를 적용하고, validation/test 데이터는 원본 분포 그대로 평가한다.

저장되는 결과는 다음과 같다.
1. `proposed_mlp_final_model.joblib`: 최종 학습된 제안 모델
2. `proposed_mlp_selected_config.json`: 최종 선택된 MLP 모델 설정
3. `proposed_mlp_metrics.csv`: validation/test 성능 평가 결과
4. `proposed_mlp_cv_summary.csv`: 후보 MLP 설정별 5-Fold 평균 성능 요약



### 5-1. 최종 모델 선택 및 학습

- `selected_config.json`이 있으면 저장된 설정을 사용한다.
- 없으면 CV 요약표에서 가장 위에 있는 후보 모델을 선택한다.
- 선택된 설정으로 전체 train 데이터에 대해 최종 모델을 다시 학습한다.


In [33]:
if can_reuse_cv_config:
    selected = cached_selected_config
    best_config = selected_config_to_model_config(selected)
    best_model_name = best_config['name']
    print(f'selected_config 기준으로 CV를 생략하고 최종 모델을 학습합니다: {best_model_name}')
else:
    best_summary = cv_summary.iloc[0]
    best_model_name = best_summary['model']
    best_config = MODEL_CONFIG_BY_NAME[best_model_name]
    selected = build_selected_config(best_config, best_summary)
    print(f'CV 결과 기준으로 최종 모델 설정을 선택합니다: {best_model_name}')

final_model = make_model(best_config, random_state=SEED)
final_model.fit(X_train_all, y_train_all)

val_prob = final_model.predict_proba(X_val)[:, 1]
test_prob = final_model.predict_proba(X_test)[:, 1]


selected_config 기준으로 CV를 생략하고 최종 모델을 학습합니다: mlp_h32_tanh_sgd_lr0.001_alpha0.01_bs64_pca0p9_smote5


### 5-2. Threshold 탐색

- 5-Fold CV에서 선택된 MLP 구성으로 최종 모델을 학습한 뒤, validation 데이터에서 threshold를 다시 조정한다.
- 기본 threshold는 0.5이다.
- 너무 낮은 기준으로 이벤트 리뷰를 과하게 예측하지 않도록, threshold 후보는 0.5 이상으로 제한한다.
- 0.5 이상 후보 중 validation F1-score가 가장 높은 threshold를 최종 threshold로 사용한다.

In [27]:

MIN_TUNED_THRESHOLD = 0.5

precisions, recalls, thresholds = precision_recall_curve(y_val, val_prob)

if len(thresholds) == 0:
    best_threshold = MIN_TUNED_THRESHOLD
    threshold_candidates = pd.DataFrame()
else:
    f1s = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
    threshold_candidates = pd.DataFrame({
        'threshold': thresholds,
        'precision': precisions[:-1],
        'recall': recalls[:-1],
        'f1': f1s,
    })
    threshold_candidates = threshold_candidates[threshold_candidates['threshold'] >= MIN_TUNED_THRESHOLD]

    if threshold_candidates.empty:
        best_threshold = MIN_TUNED_THRESHOLD
    else:
        best_threshold = float(
            threshold_candidates.sort_values('f1', ascending=False).iloc[0]['threshold']
        )

threshold_rank = threshold_candidates.sort_values('f1', ascending=False).head(10).reset_index(drop=True)
threshold_rank.insert(0, 'rank', threshold_rank.index + 1)
threshold_rank


,rank,threshold,precision,recall,f1
0,1,0.500871,0.386646,0.527542,0.446237
1,2,0.500368,0.386047,0.527542,0.445837
2,3,0.500878,0.385692,0.525424,0.444843
3,4,0.501094,0.384735,0.523305,0.443447
4,5,0.502488,0.385580,0.521186,0.443243
5,6,0.502949,0.386435,0.519068,0.443038
6,7,0.502354,0.384977,0.521186,0.442844
7,8,0.502829,0.385827,0.519068,0.442638
8,9,0.501857,0.384375,0.521186,0.442446
9,10,0.502783,0.385220,0.519068,0.442238



### 5-3. 성능 계산 및 결과 저장

- validation에서 선택한 threshold로 validation/test 성능을 계산한다.
- 최종 모델, 선택 설정, 성능표를 저장한다.
- `selected_config.json`은 이후 실행에서 CV 탐색을 건너뛰기 위한 최적 설정 캐시로도 사용한다.
- test 성능은 모델 선택에 사용하지 않고 최종 참고 지표로만 저장한다.


In [28]:

def metric_row(split: str, y_true, prob, threshold: float) -> dict:
    pred = (prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    return {
        'split': split,
        'threshold': float(threshold),
        'f1': float(f1_score(y_true, pred)),
        'pr_auc': float(average_precision_score(y_true, prob)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, prob)),
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
    }


metrics = pd.DataFrame([
    metric_row('validation_default_0_5', y_val, val_prob, 0.5),
    metric_row('validation_tuned_min_0_5', y_val, val_prob, best_threshold),
    metric_row('test_default_0_5', y_test, test_prob, 0.5),
    metric_row('test_tuned_min_0_5', y_test, test_prob, best_threshold),
])

selected = {
    **selected,
    'min_threshold_for_tuning': MIN_TUNED_THRESHOLD,
    'best_threshold_from_validation': best_threshold,
}

metrics.to_csv(PROPOSED_MLP_OUTPUTS['metrics_csv'], index=False, encoding='utf-8-sig')

with open(PROPOSED_MLP_OUTPUTS['selected_config'], 'w', encoding='utf-8') as f:
    json.dump(selected, f, ensure_ascii=False, indent=2)

model_bundle = {
    'model': final_model,
    'feature_cols': feature_cols,
    'emb_cols': emb_cols,
    'meta_cols': meta_cols,
    'target_col': TARGET_COL,
    'selected_config': selected,
    'input_type': 'tabular_raw',
    'default_threshold': 0.5,
    'best_threshold': best_threshold,
}
joblib.dump(model_bundle, PROPOSED_MLP_OUTPUTS['model'])


['outputs/proposed_mlp_final_model.joblib']

### 5-4. 결과 표 출력

- 저장된 성능 결과를 노트북에서 바로 읽을 수 있도록 표 형태로 출력한다.

In [36]:
selected_display = pd.DataFrame([{
    '선택 모델': selected['selected_model'],
    '은닉층 구조': str(tuple(selected['hidden_layer_sizes'])),
    'optimizer': selected['solver'],
    'learning_rate': selected['learning_rate_init'],
    'L2 alpha': selected['alpha'],
    'activation': selected['activation'],
    'batch_size': selected['batch_size'],
    'PCA 보존율': selected['pca_n_components'],
    'SMOTE k_neighbors': selected['smote_k_neighbors'],
}])

threshold_display = pd.DataFrame([{
    '기준': 'validation F1 기준',
    '최소 threshold': selected['min_threshold_for_tuning'],
    '선택된 threshold': selected['best_threshold_from_validation'],
    '설명': '0.5 이상 후보 중 validation F1-score가 가장 높은 threshold'
}])

metrics_display = (
    metrics[metrics['split'].isin([
        'validation_tuned_min_0_5',
        'test_tuned_min_0_5',
    ])]
    .reset_index(drop=True)
    .rename(columns={
        'split': '평가 데이터',
        'threshold': 'threshold',
        'f1': 'F1',
        'pr_auc': 'PR-AUC',
        'precision': 'Precision',
        'recall': 'Recall',
        'roc_auc': 'ROC-AUC',
        'tn': 'TN(일반→일반)',
        'fp': 'FP(일반→이벤트)',
        'fn': 'FN(이벤트→일반)',
        'tp': 'TP(이벤트→이벤트)',
    })
    .replace({
        '평가 데이터': {
            'validation_tuned_min_0_5': 'Validation',
            'test_tuned_min_0_5': 'Test',
        }
    })
    .round({
        'threshold': 4,
        'F1': 4,
        'PR-AUC': 4,
        'Precision': 4,
        'Recall': 4,
        'ROC-AUC': 4,
    })
)

print('선택된 제안 모델')
display(selected_display)

print('설정된 threshold')
display(threshold_display)

print('Validation/Test 결과')
display(metrics_display)

선택된 제안 모델


,선택 모델,은닉층 구조,optimizer,learning_rate,L2 alpha,activation,batch_size,PCA 보존율,SMOTE k_neighbors
0,mlp_h32_tanh_sgd_lr0.001_alpha0.01_bs64_pca0p9...,"(32,)",sgd,0.001,0.01,tanh,64,0.9,5


설정된 threshold


,기준,최소 threshold,선택된 threshold,설명
0,validation F1 기준,0.5,0.500871,0.5 이상 후보 중 validation F1-score가 가장 높은 threshold


Validation/Test 결과


,평가 데이터,threshold,F1,PR-AUC,Precision,Recall,ROC-AUC,TN(일반→일반),FP(일반→이벤트),FN(이벤트→일반),TP(이벤트→이벤트)
0,Validation,0.5009,0.4462,0.3850,0.3866,0.5275,0.5475,459,395,223,249
1,Test,0.5009,0.4405,0.4029,0.3983,0.4926,0.5547,502,352,240,233
